<a href="https://colab.research.google.com/github/tashsentner-commits/data-project/blob/main/data_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Project: _your title_

In Colab, use File → Upload notebook, then run the setup cell. After that, keep the notebook in your
`data-project` repository with **File › Save a copy in GitHub**.

**Name:** _your name_ &middot; **Repository:** _https://github.com/your-username/data-project_
**Data set:** _menu number and name, or your approved data set_
**Source and licence:** _paste the attribution line from the README_

One notebook, two sections. Cells marked **[A]** are graded in DATA-3125 (Section A, 30 marks).
Cells marked **[B]** are graded in DATA-3130 (Section B, 30 marks). The handout has the full
instructions and the rubrics are attached to each course's dropbox.

The setup cell downloads every menu data set into a `data` folder (about 87 MB, so give it a minute
or two) and checks each file arrived complete. Set 16 has no course copy: its data comes by code.

In [ ]:
#@title Setup: run this cell first
# Downloads this lab's data files (skipping any you already have), checks that
# each one arrived complete, and installs any extra packages. Same in every lab.
import hashlib, html, os, re, urllib.error, urllib.parse, urllib.request

DATA = [
    # (where the file goes, its Google Drive file id, its sha256 fingerprint)
    ("data/climate_daily_charlottetown_2012.csv", "17MXTHN2U5H8ucEtDmDr4zsKrzt19gu5r", "fa34d9c42a8b3e8ad7a68cd4e5afad208021acce4e6b9626bbdacfb1cb5af146"),
    ("data/climate_daily_charlottetown_2013.csv", "1YwMx3XAhTLv2ITXvVwKE1NTr8OHNid5i", "156435346fc2cb88f3a5a36f70f741581f344363ff6c61eb93d56fc8eac8d23d"),
    ("data/climate_daily_charlottetown_2014.csv", "1VQNbe3CSlBm86mxpSfuXVkLtpS2NLzxo", "4d3264e9f2d5f3a0282576b49bffff3f0ee8b82900fee88d03c88c6dce26e68d"),
    ("data/climate_daily_charlottetown_2015.csv", "1C1s5uCi7sPvsdFkywR0t1vxw7LBy3fpQ", "4af298103cb4a96c36e4178a9239b759da050042b555bce3c961dac562fd6bd8"),
    ("data/climate_daily_charlottetown_2016.csv", "1-1xyl_EBLee_afjtFoaRNNTCl88ghHVL", "176bd66ff687716177a407850a3e98e89a420b45e8f92981b39617440b014487"),
    ("data/climate_daily_charlottetown_2017.csv", "1xWXjl7TK7vVf-UofZZb30kk72hCLbREQ", "78636d6fef3655195dba4990b89da5cd23706033054a0197bb47f5250bc1cc2a"),
    ("data/climate_daily_charlottetown_2018.csv", "1Az_K9n2DIfQcEEk5llNw6eO993MyijDC", "fbd113a63ac0ce26242726ee759c2221a9ccc3997d06a95df3fbf1556272373d"),
    ("data/climate_daily_charlottetown_2019.csv", "1fmi8XuaQarZUs-aBgzvfo6tOXejN-nWg", "b7896ac4feb5e600f58e36388115fb53e2de451c8e092a73a3cd8a713275df35"),
    ("data/climate_daily_charlottetown_2020.csv", "1lrvo_C6uBaCGCFQaDoLKQjhYLRbLJ8Wv", "c1408dae1aa1df16b1a338e36f940b027d3efb24751a38da72be2bde47e260fd"),
    ("data/climate_daily_charlottetown_2021.csv", "1l8_AJtb9Bp-L7wITMniM_AZwyWh8LJhY", "b9894379db958035ada8f5593539ff5a5d54309c2d3a48f9c51708412ed51aca"),
    ("data/climate_daily_charlottetown_2022.csv", "1tZttpu38cAzNegh44D_GObqxPY4V0Sur", "1adc8ab67e70270f59c888bd0d147f57ce7ebea19fb37104c401bac4fb604d08"),
    ("data/climate_daily_charlottetown_2023.csv", "12ub1WuE1yIjo6nNBl3hIYIss4r56rWyX", "d8afb239623e0e09e1a6aa29efeff28c9e0fbbaf535ab5219d104f999fd485d1"),
    ("data/climate_daily_charlottetown_2024.csv", "1t7_IifHu25ZWHla2lfbZXTJ2vIsvfRMI", "0f52ac9249ea1915efc8da2933e01ba42de1337f6f46c8c6faa23edfba88fb7b"),
    ("data/climate_daily_charlottetown_2025.csv", "1GIDr6ZpHjNnwGY3ik1ZsH_8T4vmbAStz", "b7372b4e1b88d7b3c8ba00ab75aa8dcfffff3733b61d5b54c1c6b61fdcaf7f6a"),
    ("data/my2012-2026-battery-electric-vehicles.csv", "1N5jDiAXH7lnfyEj0vdsgY9GJY3IlLISj", "6330da2fa014c0eb7e343c0a082c67da54eaecedfa930f5a4b54217e522ca55d"),
    ("data/my2012-2026-plug-in-hybrid-electric-vehicles.csv", "14CmzebHVtx2aut4KuefZU2hZ7oTvSdnc", "e90ed83075a4dd808aae4ef660662495f0b30ed1d3739d1c51eb367a54c303e2"),
    ("data/my2015-2024-fuel-consumption-ratings.csv", "1iPQ8vHkEhTEfOTb_lLk9khV2h0KaDioe", "2279197fbdba916a1e4f94b78220d93e26ab7d73043f8a028061b27bd88933e4"),
    ("data/my2025-fuel-consumption-ratings.csv", "1sW2uF-JzqZNUsWhmZtIBnXkgaUF8TYpP", "03b227ae4ca027a54699541bf357c48ac18f743f1f9af21e38be99de8826d988"),
    ("data/my2026-fuel-consumption-ratings.csv", "1s4gNpAHDxzlIEyQt72WjT7Th6s_Ii_AU", "69846a63485eceb88e4988c29e10abb997004af3118b74d5388333e2171f3751"),
    ("data/statcan_18100245_retail_prices_atlantic.csv", "1IGqcS1vKrpwAXt79IxJOexV_mRTxdyZK", "0885783af58041b8e0dc7cf3dca3b422458801aef50643bdb3e02b13f3913975"),
    ("data/statcan_18100001_gasoline_fuel_oil_prices.csv", "1PGViLR9doK9HUscnz4KrPgnazCxVShai", "b04150c423c3fecad188bbb1e9ccf2fcf89f15372508f579cd1393e1c5396a18"),
    ("data/tourism_ns_visitation.csv", "1idXt-EjSPfhyh5qJGl5omHMo663uZh8i", "d4b1a2d715466772617e2e104e60e2f8374646e5b70a90c12b2fd2f9b043f37b"),
    ("data/pei_precipitation_monitoring.csv", "1t4z8acpe_EhEvEjE0D_835OS3hRY2O_2", "a2dc3e1449fddce6f715525a4c81080ed93858b2d5e9bd2c3660aefab2fee3c1"),
    ("data/pei_surface_water_quality_hillsborough_west.csv", "13GTMDd-sW3O5ymfRJuAUwhQN1UTrF2O5", "9fe3e50d88fbab29b06e47c416c24a52fabdc93ba2eba0df8eb2c108b3f2b3b1"),
    ("data/ns_crime_incidents_rates.csv", "1AbwbfwlxrmNJAZvUOpXu-Y7lNabl4ll4", "5e37f6e67b2f456a2813fab039ba777e49f9fa8643a882c7f1a9552cfc93d2d1"),
    ("data/ns_crime_severity_index.csv", "1VkJ5q3u4cZo9Ggk-pKQeccC_LFbXOQrB", "2613ef71f8467d5a1a3fada5ff74f595dbd942a2cfa80b6f7e685eb972ffc1c2"),
    ("data/owid-co2-codebook.csv", "1TIqut2hJk0_ofawv_oDKtLfOihBiyMh0", "33b4f5e00efd58c7b83863f736beba1af4df946b43642b0400c3ec38648e0e8e"),
    ("data/owid-co2-data.csv", "18WPdtGEaDTfQzXXEjFehK-kr_4MrQIqR", "7f78e2b218ce4bb8c538bbec04fdc9a7982e8d40bff972e650df603899edd5f6"),
    ("data/former_names.csv", "1MHv0xF4N0iu7UcWcplgBM3tJv-E0BWUr", "2d57f59a3ac13b24c2fcf208ced5a38710529d122da38929838d0980b002677e"),
    ("data/goalscorers.csv", "19rGEAvqBnM_vcPWO7a4mXaYo8TIXonV4", "38cbc8007fb4aef7dd9f57e354623dca2abaf03a859530565ccf368e86a9f614"),
    ("data/results.csv", "1YQBiX5LndYFyG3o3OIqq66lq69hX1QOg", "df35268f8fc341ff7fb93d448b4e40356676ac35300a6b4461fd199a99ac1514"),
    ("data/shootouts.csv", "13r7HK4Fhj6i7Tc0oxDbYsq5KcIBXVmy5", "2acdc95fdad14f5f17200150a1a4a5fa6c2ccf57d31c8b4f61a2c7d9c0493113"),
    ("data/hotel_bookings.csv", "1xA2KY52GskVZLcROqEHv3BhttIgOWjVL", "7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06"),
    ("data/netflix_titles.csv", "1Vbj76yXCB5eXt5Upgfabh2tUSPmghA9c", "df1f4ad2027a5a14c3a33932ef0d4054565ff88adf92b7f263601b70fdc6f3f3"),
    ("data/googleplaystore.csv", "1yvdjpux0LOkoCiRfamzhT4y8wGnAC5Gb", "3e438f48161961933d26e99a8d9fc8ed79edfaa9fb34f8838e1ab4ec7a9fab91"),
    ("data/googleplaystore_user_reviews.csv", "1D7TulYgOdJ9n3hYCp-K2mj-sOELU-CiU", "5074250eaf0722db4a5707f0b06dcdf3663310348761ccaab01ec6f94411269b"),
    ("data/AB_NYC_2019.csv", "1lo9XvXZoSvc4aAcA3I37FDBfrKiKknOy", "e420db40ff10fcb40efc1b5b1648ee0b18a48f4e4537155cecc59fe95d18783a"),
]
DRIVE = "https://drive.google.com/uc?export=download&id="
OFFLINE = ": could not reach Google Drive. Check your internet connection and run this cell again."
NOT_SHARED = (": the file isn't shared or Drive is busy. Wait a few minutes and run this "
              "cell again; if it keeps failing, tell your instructor.")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def download(file_id, path):
    name, url = os.path.basename(path), DRIVE + file_id
    for _ in range(3):
        try:
            with urllib.request.urlopen(url, timeout=120) as response:
                headers = response.headers
                if ("attachment" in headers.get("Content-Disposition", "")
                        or "text/html" not in headers.get("Content-Type", "")):
                    with open(path + ".part", "wb") as f:
                        for block in iter(lambda: response.read(1 << 20), b""):
                            f.write(block)
                    os.replace(path + ".part", path)
                    return
                page, here = response.read().decode("utf-8", "replace"), response.geturl()
        except urllib.error.HTTPError:
            break  # Drive answered, but not with the file (not shared, removed or busy)
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            raise RuntimeError(name + OFFLINE) from None
        # A large file first gets a "can't scan for viruses" page; follow its download form.
        form = re.search(r'<form[^>]*id="download-form"[^>]*>(.*?)</form>', page, re.S)
        if not form:
            break
        action = re.search(r'action="([^"]+)"', form.group(0)).group(1)
        fields = dict(re.findall(r'name="([^"]+)"\s+value="([^"]*)"', form.group(1)))
        url = urllib.parse.urljoin(here, html.unescape(action)) + "?" + urllib.parse.urlencode(fields)
    raise RuntimeError(name + NOT_SHARED)


for path, file_id, expected in DATA:
    name = os.path.basename(path)
    if not (os.path.exists(path) and sha256(path) == expected):
        if file_id is None:
            raise RuntimeError(name + " has no download link yet: this lab is still being "
                               "prepared. Tell your instructor.")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        download(file_id, path)
        if sha256(path) != expected:
            os.remove(path)
            raise RuntimeError(name + " did not download completely. Run this cell again.")
    print(name + " ready")

In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# The data-quality log. Call note(...) after every fix in A2 and A3; A5 shows the table.
quality_log = []


def note(problem, where, action, rows_affected, reason):
    """Add one row to the data-quality log."""
    quality_log.append({"problem": problem, "where": where, "action": action,
                        "rows_affected": rows_affected, "reason": reason})


# Your menu set number as text, for example "07", or "own" for set 16 or an approved data set.
DATASET = ""

print("Libraries are ready")

## Proposal (checkpoint, Tue Nov 24)

Answer in this cell, save to GitHub, and upload to the checkpoint dropbox. If your plan changes
later, update the answers and add a line saying what changed.

1. **Data set:** _menu number and name, or source, licence, rows and columns of your own_
2. **Files and rows:** _which files, and what one row means in each_
3. **Merging:** _which two tables (your B1 table may be one), on which key, and one problem you expect with the key_
4. **Collecting by code:** _URL or API, key needed or not, the linking column, and whether a test request from Colab worked_
5. **Questions:** _Q1, Q2, Q3_
6. **Modelling:** _a target column and two features you might build_
7. **Repository:** _URL of your private `data-project` repository; instructor invited?_

## A1. Data set and documentation [A]

Load each file you use. Under the code, list each table with its row count and one sentence on what
one row means.

In [ ]:
# raw_x = pd.read_csv("data/....csv")

if DATASET == "15":
    # Hosts are private people, and the project uses no personal information, so host_name is dropped on load.
    raw_listings = pd.read_csv("data/AB_NYC_2019.csv").drop(columns=["host_name"])

_Table, rows, and what one row means:_

## B1. Collect a table by code [B]

Get one more table from the web with code: an API (`requests`), a web page (Beautiful Soup), or a
scripted download of a published file. Check the request worked before you use the result. Write
down the source URL, its licence or terms, and the date you collected it. Save a copy to
`data/collected_<name>.csv`. To put it in your repository, download it from Colab (the **Files**
pane, then **Download**) and upload it on GitHub with **Add file › Upload files** into a `data/`
folder.

If your source needs a key, store it in Colab's **Secrets** panel and read it with `userdata.get`,
as in DATA-3130 Lab 8, part E ("Keep a key out of your notebook"). Use the name you gave the
secret. Never type a key into a cell.

In [ ]:
import requests
# from google.colab import userdata   # only if your source needs a key
# key = userdata.get("YOUR_SECRET_NAME")

# response = requests.get(URL, params={...}, timeout=30)
# response.raise_for_status()
# collected = pd.DataFrame(...)
# os.makedirs("data", exist_ok=True)
# collected.to_csv("data/collected_<name>.csv", index=False)

_Source URL, licence or terms, date collected, and the column that links it to your data:_

## B4. A small class or function, and its tests [B]

Write one small class (or function) that your project really uses, for example a cleaner for one
messy column, or the code that calls your API. Give it a docstring. Use it later in the notebook.

In [ ]:
class ColumnCleaner:
    """Replace this example with your own class: say what it takes and what it returns."""

    def __init__(self):
        pass

### Tests [B]

At least three `assert` cases, one of them an edge case (a blank, a zero, unexpected text). The
cell prints "no tests yet" until you write an `assert`, and "all tests passed" once they all pass.

In [ ]:
def test_my_code():
    # assert ... == ..., "what should have happened"
    # assert ... == ..., "what should have happened"
    # assert ... == ..., "edge case: what should have happened"
    pass


import inspect

try:
    has_tests = any(line.strip().startswith("assert")
                    for line in inspect.getsource(test_my_code).splitlines())
except OSError:  # source not available (outside Colab or Jupyter)
    has_tests = True
if has_tests:
    test_my_code()
    print("all tests passed")
else:
    print("no tests yet")

## A2. Profile, then clean [A]

### Profile before you change anything
For every table: shape, types, missing values per column, ranges, duplicate rows, and the distinct
values of the key columns. Under the code, say what surprised you.

In [ ]:
# raw_x.shape, raw_x.dtypes, raw_x.isna().sum(), raw_x.describe(), raw_x.duplicated().sum()

### Clean
Work on a copy (`clean_x = raw_x.copy()`) so the raw tables stay as loaded. For each problem: what
you found, what you did, why, and the row count before and after. Call `note(...)` after each fix.

In [ ]:
# clean_x = raw_x.copy()
# before = len(clean_x)
# ... one fix ...
# note("problem", "table and column", "what you did", before - len(clean_x), "why")

## A3. Transform and combine [A]

Reshape or retype at least one table (parse dates, pivot or melt, decode codes). Stacking files of
the same shape with `pd.concat` is a useful preparation step, but the graded combine is a **merge**
of two different tables on a key. If your menu set has only one kind of table, merge it with your
B1 collected table. Before the merge: check the key is unique where it should be, and find the keys that do not
match (`indicator=True` or an anti-join). After it: compare row counts and explain any change, and
say why you chose that join type.

In [ ]:
# key checks before the merge

In [ ]:
# the merge, with row counts before and after

## B2. Three questions, answered with pandas [B]

Each question should need more than one number to answer. Show the result, then answer in one
sentence that quotes a number from it.

### Q1: _your question_

_Answer:_

### Q2: _your question_

_Answer:_

### Q3: _your question_

_Answer:_

## B3. Four charts [B]

Use at least two of matplotlib, seaborn and Plotly. Each title states the point the chart makes,
both axes are labelled with units, and one sentence under the chart says what to notice.

### Chart 1

_What to notice:_

### Chart 2

_What to notice:_

### Chart 3

_What to notice:_

### Chart 4

_What to notice:_

## A4. Features and a train/test split [A]

Name the target column a model could predict. Build at least three new features of at least two
kinds (date parts, ratios or differences, bins, encoded categories, scaled values) and give one line
on why each could help. Split with a fixed `random_state`, show the shapes, and fit any scaler or
encoder on the training rows only.

_Target, and what a model would predict:_

In [ ]:
# features

In [ ]:
from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## A5. Data-quality notes [A]

The log below is built from your `note(...)` calls. Check that its counts match what your cells
printed.

In [ ]:
pd.DataFrame(quality_log)

### What remains
At least two problems you did not fix, and how each could affect your results.

1. _problem, and its effect_
2. _problem, and its effect_

## B5. Git history [B]

Before you submit, in Git Bash in your `data-project` folder, push anything left (`git push`),
then run:

```
git pull
git log --graph --oneline --all
```

Paste the whole output into the cell below, replacing the placeholder, then save a final copy to
GitHub. That last save is the one commit the pasted log cannot show. The log should show the work
building up over the project weeks: several commits, each a real step with a message that says what
changed, and at least one branch merged into `main`. One large upload near the deadline does not
show this. Your README on GitHub should give the
project title, your name, the data sources, and how to run this notebook.

**My Git history:**

```
paste the output of git log --graph --oneline --all here
```

## AI use

Say which AI tools you used, for what, and which cells they affected. If you used none, write
"None".

_AI use:_

## Before you submit

1. **Runtime › Restart session and run all**, and fix anything that stops with an error.
2. Paste your Git history above, then save a final copy to GitHub.
3. **File › Download › Download .ipynb**, named `Lastname_Firstname_DataProject.ipynb`.
4. Upload the same file to the Data Project dropbox in D2L (SAM) for DATA-3125 and for DATA-3130,
   by 11:59 PM on Mon Dec 14, with your repository URL in the submission comment.